# E03 — o que o vigia vê

O capítulo anterior mediu a latência de um vigia contra o topo do mercado, porque no mercado
**não se sabe quando o mundo mudou**. Aqui a mudança é posta de propósito: o banco de provas
devolve séries com forma e data conhecidas, e a latência vira uma medida de verdade.

**As formas** (mudanca.py): o degrau (a oscilação dobra ou octuplica de uma vez), a rampa (a
mesma oscilação, chegando ao longo de meses), a deriva (a média se desloca todo dia) e o
mundo parado, que é o controle sem o qual nenhuma das outras vale.

**Os instrumentos**, os dois calibrados no mesmo orçamento medido em mundo parado:

1. o **vigia do capítulo 2** — a contagem de rompimentos do corte em blocos de 60 dias;
2. o **centro** (centro.py) — a média da janela medida em erros-padrão dela mesma.

**Simulação não vira resultado sobre o mundo** (AGENTS.md §8.5): o que se mede aqui é uma
propriedade dos instrumentos — quanto cada um leva para ver uma mudança de forma e tamanho
declarados. Nada disto é uma afirmação sobre o mercado.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo, resultado em
lab/resultados/E03_formas.json, figura em .pdf e .png.

In [1]:
# <- brinque com: SERIE_ANOS, QUANDO, JANELA, BLOCO, LIMIAR, LIMIAR_CENTRO, MUNDOS, CASOS
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import centro, graficos, mudanca, promessa, vigia

RAIZ = Path.cwd()
ANOS_SERIE = 40
DIAS_UTEIS = 252
N = ANOS_SERIE * DIAS_UTEIS          # o tamanho de cada mundo
QUANDO = N // 2                      # a mudança entra no meio, e a data é conhecida
JANELA = 252                         # o corte do capítulo 1
BLOCO = 60                           # o bloco do capítulo 1
LIMIAR = 8                           # o limiar do vigia do capítulo 2
LIMIAR_CENTRO = 3.0                  # o limiar do centro, em erros-padrão
MUNDOS = 600                         # mundos parados, para o orçamento
CASOS = 120                          # mundos por forma, para a latência
HORIZONTE = 500                      # dias depois da mudança em que se procura o alarme
SEMENTE = 19

index = pd.date_range("1985-01-02", periods=N, freq="B")
MUDANCA = index[QUANDO]
SORTEIO = np.random.default_rng(SEMENTE)
BANCO = {"estavel": mudanca.estavel, "degrau": mudanca.degrau,
         "rampa": mudanca.rampa, "deriva": mudanca.deriva}
print("frevolab %s | %d dias (%.0f anos) | a mudanca entra no dia %d, em %s" % (
    frevolab.VERSAO, N, N / DIAS_UTEIS, QUANDO, MUDANCA.date()))

frevolab 0.1.0 | 10080 dias (40 anos) | a mudanca entra no dia 5040, em 2004-04-28


## O orçamento dos dois instrumentos

In [2]:
# O orçamento: com que frequência cada instrumento soa num mundo que nunca muda.
T_CANDIDATOS = (7, 8, 9, 10)
CENTRO_CANDIDATOS = (2.5, 3.0, 3.5, 4.0)
episodios = {("vigia", t): [] for t in T_CANDIDATOS}
episodios.update({("centro", c): [] for c in CENTRO_CANDIDATOS})

for _ in range(MUNDOS):
    calmo = pd.Series(mudanca.estavel(N, SORTEIO), index=index)
    contagem = promessa.conta_em_blocos(promessa.violacoes(calmo, JANELA), BLOCO)
    estatistica = centro.media_padronizada(calmo, BLOCO)
    for t in T_CANDIDATOS:
        episodios[("vigia", t)].append(len(vigia.alarmes(contagem, t)))
    for c in CENTRO_CANDIDATOS:
        episodios[("centro", c)].append(
            len(vigia.alarmes((estatistica.abs() >= c).astype(float), 1)))

ANOS = N / DIAS_UTEIS
print("%8s %10s %14s" % ("instrumento", "limiar", "anos por alarme"))
for chave in sorted(episodios, key=lambda k: (k[0], k[1])):
    media = np.mean(episodios[chave]) / ANOS
    print("%8s %10s %14s" % (chave[0], chave[1], ("%.1f" % (1 / media)) if media else "nunca"))
print()
print("escollhidos: vigia no limiar %d | centro em %.1f" % (LIMIAR, LIMIAR_CENTRO))

instrumento     limiar anos por alarme
  centro        2.5            1.3
  centro        3.0            4.5
  centro        3.5           17.3
  centro        4.0           77.4
   vigia          7            1.7
   vigia          8            4.4
   vigia          9           13.0
   vigia         10           42.6

escollhidos: vigia no limiar 8 | centro em 3.0


## A latência, forma por forma

In [3]:
# A latência de cada instrumento, forma por forma, com a data da mudança conhecida.
CASOS_POR_FORMA = (
    ("estavel", {}),
    ("degrau", {"fator": 2.0}),
    ("degrau", {"fator": 8.0}),
    ("rampa", {"fator": 2.0}),
    ("rampa", {"fator": 8.0}),
    ("deriva", {"passo": -0.0005}),
    ("deriva", {"passo": -0.002}),
)
rotulos, latencias, sem_alarme = [], [], []
for nome, opcoes in CASOS_POR_FORMA:
    completas = dict(opcoes, **({} if nome == "estavel" else {"quando": QUANDO}))
    lat = {"vigia": [], "centro": []}
    for _ in range(CASOS):
        serie = BANCO[nome](N, SORTEIO, **completas)
        r = pd.Series(serie, index=index)
        contagem = promessa.conta_em_blocos(promessa.violacoes(r, JANELA), BLOCO)
        estatistica = centro.media_padronizada(r, BLOCO)
        for chave, sinal in (("vigia", contagem >= LIMIAR),
                             ("centro", estatistica.abs() >= LIMIAR_CENTRO)):
            achado = vigia.latencia(sinal, MUDANCA)
            if achado <= HORIZONTE:
                lat[chave].append(achado)
    rotulos.append("%s %s" % (nome, opcoes.get("fator", opcoes.get("passo", ""))))
    latencias.append(lat)
    sem_alarme.append({k: 100 * (1 - len(v) / CASOS) for k, v in lat.items()})

print("%16s %18s %12s %18s %12s" % ("forma", "vigia: mediana", "nada ve", "centro: mediana", "nada ve"))
for rotulo, lat, sem in zip(rotulos, latencias, sem_alarme):
    print("%16s %18s %12s %18s %12s" % (
        rotulo,
        ("%.0f d" % np.median(lat["vigia"])) if lat["vigia"] else "nunca",
        "%.0f%%" % sem["vigia"],
        ("%.0f d" % np.median(lat["centro"])) if lat["centro"] else "nunca",
        "%.0f%%" % sem["centro"]))

           forma     vigia: mediana      nada ve    centro: mediana      nada ve
        estavel               251 d          64%              250 d          75%
      degrau 2.0               30 d           0%              250 d          70%
      degrau 8.0               11 d           0%              315 d          78%
       rampa 2.0              117 d           1%              255 d          82%
       rampa 8.0               46 d           0%              262 d          78%
  deriva -0.0005              218 d          61%              227 d          78%
   deriva -0.002               82 d          48%              164 d           9%


## O que sobra depois que a memória se atualiza

In [4]:
# O que sobra do vigia depois que a memória se atualiza: o corte adaptativo contra o fixo.
T_PARADOS = 2000
formas = (("degrau", {"fator": 8.0}), ("rampa", {"fator": 8.0}),
          ("deriva", {"passo": -0.002}), ("estavel", {}))
print("%10s %20s %20s %18s" % ("forma", "adaptativo (depois)", "fixo (depois)", "fixo antes"))
estado = {}
for nome, opcoes in formas:
    completas = dict(opcoes, **({} if nome == "estavel" else {"quando": QUANDO}))
    adaptativo, fixo, fixo_antes = [], [], []
    for _ in range(60):
        serie = BANCO[nome](N, SORTEIO, **completas)
        r = pd.Series(serie, index=index)
        c_adapt = promessa.conta_em_blocos(promessa.violacoes(r, JANELA), BLOCO)
        corte = float(promessa.corte(r, JANELA).iloc[JANELA])
        c_fixo = promessa.conta_em_blocos(r < corte, BLOCO)
        adaptativo.append(c_adapt.iloc[QUANDO + JANELA:].tail(T_PARADOS).mean())
        fixo.append(c_fixo.iloc[QUANDO + JANELA:].tail(T_PARADOS).mean())
        fixo_antes.append(c_fixo.iloc[JANELA:QUANDO].mean())
    estado[nome] = (float(np.mean(adaptativo)), float(np.mean(fixo)), float(np.mean(fixo_antes)))
    print("%10s %20.2f %20.2f %18.2f" % (nome, np.mean(adaptativo), np.mean(fixo), np.mean(fixo_antes)))
print()
print("a promessa admite %.1f rompimentos em %d dias" % (BLOCO * 0.05, BLOCO))

     forma  adaptativo (depois)        fixo (depois)         fixo antes


    degrau                 3.09                25.03               3.18


     rampa                 3.08                25.14               3.04


    deriva                 3.06                 4.56               3.09


   estavel                 3.08                 3.34               3.34

a promessa admite 3.0 rompimentos em 60 dias


## A lei da deriva

In [5]:
# A lei da deriva: quanta memoria uma regra de tres erros-padrao exige para ver a deriva.
LEI = (("forte", 0.002, (60, 125, 225, 400)), ("leve", 0.0005, (400, 1000, 2400, 3600)))
MUNDOS_LEI, CASOS_LEI = 200, 60
TAXA_DIARIA = 1.0 / (4.5 * DIAS_UTEIS)      # o mesmo orcamento, dito por dia

print("a janela exigida cresce com o quadrado de (limiar x barulho / passo)")
print("%8s %8s %12s %12s %14s %10s" % ("deriva", "janela", "lei (dias)", "t esperado", "mediana", "nada ve"))
lei_medida = {}
for nome, passo, janelas in LEI:
    for janela in janelas:
        nulos = []
        for _ in range(MUNDOS_LEI):
            calmo = pd.Series(mudanca.estavel(N, SORTEIO), index=index)
            nulos.append(centro.media_padronizada(calmo, janela).to_numpy())
        limiar_lei = centro.limiar_do_orcamento(np.concatenate(nulos), TAXA_DIARIA)
        lat = []
        for _ in range(CASOS_LEI):
            serie = mudanca.deriva(N, SORTEIO, passo=-passo, quando=QUANDO)
            t = centro.media_padronizada(pd.Series(serie, index=index), janela)
            achado = vigia.latencia(t.abs() >= limiar_lei, MUDANCA)
            if achado <= HORIZONTE:
                lat.append(achado)
        lei_medida[(nome, janela)] = (limiar_lei, lat)
        print("%8s %8d %12.0f %12.2f %14s %10s" % (
            nome, janela, (LIMIAR_CENTRO * mudanca.SIGMA_PADRAO / passo) ** 2,
            passo * np.sqrt(janela) / mudanca.SIGMA_PADRAO,
            ("%.0f d" % np.median(lat)) if lat else "nunca",
            "%.0f%%" % (100 * (1 - len(lat) / CASOS_LEI))))
print()
print("a mais leve: %.0f dias de memoria para uma deriva de %.0f%% ao ano" % (
    LEI[1][2][-1], 100 * DIAS_UTEIS * LEI[1][1]))

a janela exigida cresce com o quadrado de (limiar x barulho / passo)
  deriva   janela   lei (dias)   t esperado        mediana    nada ve
   forte       60          225         1.55          156 d        30%


   forte      125          225         2.24          223 d        17%
   forte      225          225         3.00          202 d         8%


   forte      400          225         4.00          306 d         3%
    leve      400         3600         1.00          380 d        95%


    leve     1000         3600         1.58          486 d        98%
    leve     2400         3600         2.45          nunca       100%


    leve     3600         3600         3.00          nunca       100%

a mais leve: 3600 dias de memoria para uma deriva de 13% ao ano


## As figuras

In [6]:
# Figura 1: o mundo e o vigia, para quatro formas. O preço acumulado em cima, a contagem embaixo.
FORMA_FIGURA = (("estavel", {}), ("degrau", {"fator": 8.0}),
                ("rampa", {"fator": 8.0}), ("deriva", {"passo": -0.002}))
CORES = ("#7f7f7f", "#1f4e79", "#b8860b", "#b03a2e")
serie_figura, contagem_figura = {}, {}
for nome, opcoes in FORMA_FIGURA:
    completas = dict(opcoes, **({} if nome == "estavel" else {"quando": QUANDO}))
    serie = BANCO[nome](N, SORTEIO, **completas)
    r = pd.Series(serie, index=index)
    chave = "%s %s" % (nome, opcoes.get("fator", opcoes.get("passo", "")))
    serie_figura[chave] = (100 * r.cumsum())
    contagem_figura[chave] = promessa.conta_em_blocos(promessa.violacoes(r, JANELA), BLOCO)

fig, (cima, baixo) = plt.subplots(2, 1, figsize=(9.4, 6.2), sharex=True,
                                  gridspec_kw={"height_ratios": [1, 1]})
for cor, (chave, valores) in zip(CORES, serie_figura.items()):
    cima.plot(valores.index, valores.to_numpy(), color=cor, lw=1.2, label=chave)
    baixo.plot(contagem_figura[chave].index, contagem_figura[chave].to_numpy(), color=cor, lw=1.0)
cima.axvline(MUDANCA, color="black", lw=1.0, ls=":")
cima.set_ylabel("retorno acumulado (%)")
cima.legend(frameon=False, fontsize=9, loc="lower left")
cima.grid(alpha=0.25)
baixo.axvline(MUDANCA, color="black", lw=1.0, ls=":")
baixo.axhline(LIMIAR, color="#b03a2e", ls="--", lw=1.1, label="o limiar do vigia: %d" % LIMIAR)
baixo.set_ylabel("rompimentos em %d dias" % BLOCO)
baixo.set_xlabel("ano")
baixo.legend(frameon=False, fontsize=9, loc="upper left")
baixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E03_formas", 1)
plt.close(fig)
print("figura 1 gravada")

figura 1 gravada


## Leitura visual das figuras

_(a preencher depois de olhar o .png)_

In [7]:
# Figura 2: a latência mediana de cada instrumento, forma por forma, contra o mundo parado.
posicoes = np.arange(len(rotulos))
largura = 0.38

fig, eixo = plt.subplots(figsize=(9.4, 4.4))
mediana_vigia = [np.median(l["vigia"]) if l["vigia"] else HORIZONTE for l in latencias]
mediana_centro = [np.median(l["centro"]) if l["centro"] else HORIZONTE for l in latencias]
eixo.barh(posicoes - largura / 2, mediana_vigia, largura, color="#1f4e79", label="o vigia (contagem)")
eixo.barh(posicoes + largura / 2, mediana_centro, largura, color="#b8860b", label="o centro (média em erros-padrão)")
parado = np.median(latencias[0]["vigia"])
eixo.axvline(parado, color="#7f7f7f", ls="--", lw=1.2,
             label="o mundo parado, no vigia: %.0f dias" % parado)
eixo.set_yticks(posicoes)
eixo.set_yticklabels(rotulos, fontsize=9)
eixo.set_xlabel("latência mediana (dias; a barra cheia quer dizer que ele não viu)")
eixo.legend(frameon=False, fontsize=8, loc="lower right")
eixo.grid(alpha=0.25, axis="x")
fig.tight_layout()
graficos.salvar(fig, "E03_formas", 2)
plt.close(fig)
print("figura 2 gravada")

figura 2 gravada


In [8]:
# O resultado: um grandezas para o livro citar por comando.
# Só o que o capítulo cita: medida que o livro não usa é medida morta.
CITADAS = {"parado": 0, "degrau_oito": 2, "rampa_dois": 3, "rampa_oito": 4, "deriva_forte": 6}
resultado = {
    "formas_casos": int(CASOS),
    "formas_horizonte_dias": int(HORIZONTE),
    "formas_limiar_centro": float(LIMIAR_CENTRO),
    "formas_deriva_leve_ano_pct": 100 * 252 * 0.0005,
    "formas_deriva_forte_ano_pct": 100 * 252 * 0.002,
    "formas_memoria_deriva_leve": float((LIMIAR_CENTRO * mudanca.SIGMA_PADRAO / 0.0005) ** 2),
    "formas_memoria_deriva_forte": float((LIMIAR_CENTRO * mudanca.SIGMA_PADRAO / 0.002) ** 2),
    "formas_contagem_adaptativa_degrau": estado["degrau"][0],
    "formas_contagem_fixa_degrau": estado["degrau"][1],
    "formas_contagem_parado": estado["estavel"][0],
    "formas_contagem_fixa_parado": estado["estavel"][1],
    "formas_contagem_fixa_deriva": estado["deriva"][1],
    "formas_lei_forte_dias": float(np.median(lei_medida[("forte", 225)][1]))
        if lei_medida[("forte", 225)][1] else float(HORIZONTE),
    "formas_lei_forte_nada_pct": float(100 * (1 - len(lei_medida[("forte", 225)][1]) / CASOS_LEI)),
    "formas_lei_leve_nada_pct": float(100 * (1 - len(lei_medida[("leve", 3600)][1]) / CASOS_LEI)),
}
for nome, i in CITADAS.items():
    resultado["formas_vigia_%s_dias" % nome] = (
        float(np.median(latencias[i]["vigia"])) if latencias[i]["vigia"] else float(HORIZONTE))
    resultado["formas_vigia_%s_nada_pct" % nome] = float(sem_alarme[i]["vigia"])
for nome, i in (("degrau_oito", 2), ("rampa_oito", 4), ("deriva_forte", 6)):
    resultado["formas_centro_%s_dias" % nome] = (
        float(np.median(latencias[i]["centro"])) if latencias[i]["centro"] else float(HORIZONTE))
    resultado["formas_centro_%s_nada_pct" % nome] = float(sem_alarme[i]["centro"])

caminho = Path("lab/resultados/E03_formas.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E03_formas.json gravado | 31 grandezas
